# Valora AI Agents

## Agent Architecture & Implementation Overview

This notebook focuses exclusively on the AI agent system in Valora DMPE+. It documents the agent roster, orchestrators, execution flow, data dependencies, and example API usage for agent-driven analysis.

## Multi-Agent Architecture (Agents Only)

```text
User Intent
   │
   ▼
Planner Agent ──► Plan (tasks, dependencies, parallelism)
   │
   ├─► Map Agent (spatial/zone discovery)
   ├─► AVM Agent (valuation & comps)
   ├─► Forecasting Agents (Prophet | ARIMA | Ensemble)
   ├─► Risk Agents (Market | Liquidity | Regulatory)
   └─► Explainability Agents (SHAP | PDP | Counterfactual)
   │
   ▼
Critic/Validator ──► Validate, score, and refine results
   │
   ▼
Aggregator ──► Final response + recommendations + optional map actions
```


## Agent Roster & Roles

- Spatial Intelligence
  - RasterAgent: satellite imagery-derived features and neighborhood signals
  - GraphAgent: property-network analysis, communities, influence propagation

- Valuation
  - AVMAgent: ensemble valuation, comparables, market positioning, breakdown

- Forecasting
  - ProphetAgent: seasonality-aware forecasts
  - ARIMAAgent: statistical forecasts, auto-ARIMA
  - EnsembleForecaster: combined forecast for robustness

- Risk Assessment
  - MarketRiskAgent: volatility, VaR, stress scenarios
  - LiquidityRiskAgent: days-on-market, absorption, transaction velocity
  - RegulatoryRiskAgent: compliance risks and flags

- Explainability
  - SHAPAgent: feature importances and local explanations
  - PDPAgent: partial dependence analyses
  - CounterfactualAgent: what-if scenarios and actionable deltas


## Orchestrators & Routing

- Active orchestrator: `ValoraOrchestrator` (wired in backend API)
- Enhanced orchestrator (present, optional): `EnhancedValoraOrchestrator`

Routing flow:
- Planner builds a plan from intent (tasks, order, parallel groups)
- Agent registry resolves agent names to implementations under `backend/services/agents/`
- Executor runs tasks (parallel/sequential), collects outputs
- Critic validates and scores results; errors are handled gracefully
- Aggregator synthesizes: chat response, recommendations, map actions, confidence


## Execution Workflow & Example Plan

Example plan (simplified):

```json
{
  "plan_id": "plan_2024_001",
  "tasks": [
    {"id": "t1", "agent": "map_agent", "action": "identify_zones", "params": {"city": "Bangalore", "type": "3BHK"}},
    {"id": "t2", "agent": "avm_agent", "action": "estimate", "depends_on": ["t1"], "params": {"locality": "Whitefield", "bedrooms": 3, "area_sqft": 1200}},
    {"id": "t3", "agent": "forecaster", "action": "forecast", "depends_on": ["t2"], "params": {"months": 12}},
    {"id": "t4", "agent": "risk_agent", "action": "assess", "depends_on": ["t2"], "params": {}},
    {"id": "t5", "agent": "explainability", "action": "shap_explain", "depends_on": ["t2"], "params": {}}
  ],
  "parallel_groups": [["t1"], ["t2"], ["t3","t4","t5"]]
}
```

Outputs merged into:
- final_result.chat_response
- final_result.recommendations
- final_result.analysis (valuation, forecast, risk, spatial)
- map_action (optional): center/zoom, draw polygon/buffer, show properties
- confidence (0.0–1.0)


## Agent Interface Contract

All agents expose a common async interface:

```python
async def execute(self, action: str, parameters: dict) -> dict:
    """Returns structured result dict with keys like:
    {
      "status": "success",
      "analysis": {...},
      "recommendations": [...],
      "map_action": {...},
      "confidence": 0.85,
      "errors": []
    }
    """
```

Actions by category (examples):
- AVM: estimate, compare, breakdown, forecast
- Forecast: forecast, analyze_seasonality, detect_changepoints
- Risk: assess, calculate_var, stress_test
- Explainability: shap_explain, feature_importance, counterfactual
- Spatial: identify_zones, buffer_analysis, compare_zones


## Data Dependencies (Agent-Facing)

- Processed datasets: `data/processed/` (primary source for AVM/forecast/risk)
- Models: `dmpe/models/` (pretrained artifacts when available)
- Spatial context: Mappls service (POIs, geocoding, spatial features)
- Optional libs: `prophet`, `statsmodels`, `xgboost`, `scikit-learn` (agents degrade gracefully when unavailable)

Environment:
- Backend FastAPI: http://localhost:8000
- Node proxy (optional): http://localhost:3001
- Frontend: http://localhost:3000


## API Usage (Agent Endpoints)

Primary endpoint (planner + orchestrator):

```python
import requests

resp = requests.post("http://localhost:8000/api/agent/plan", json={
  "user_id": "user_demo",
  "intent": "Find 3BHK under 1.5Cr in HSR and forecast next 12 months",
  "context": {"preferences": {"risk_tolerance": "medium"}}
})
print(resp.json())
```

Alternate multi-agent chat (if enabled):

```python
import requests

resp = requests.post("http://localhost:8000/api/multi-agent/chat", json={
  "message": "Analyze a 2BHK in Whitefield and show hotspots",
  "user_id": "user_demo"
})
print(resp.json())
```


also i can use free plans of api also

In [ ]:
Yes — you can absolutely build the VALORA-DMPE+ prototype using free plans and open-source tools. Below is a compact, practical guide: which free/OSS services to use, what to expect from their free tiers (limits + caveats), exactly which pieces of the architecture you can run for free locally, and a recommended cheap-first stack to get a functioning end-to-end prototype.

I checked current provider pages while writing this so the statements below about free tiers are accurate to available provider docs/announcements. 
Pinecone
+4
Pinecone
+4
Chroma
+4

Quick summary (one-line)

Use open-source/local tools for everything core (Postgres/PostGIS, FAISS/Chroma, MinIO, sentence-transformers) and optionally rely on managed free tiers (Pinecone starter, Hugging Face inference credits) for convenience or when you need hosted services. 
GitHub
+2
GitHub
+2

What you can run for free (and recommended providers)

1) Vector search

Chroma (open-source) — easy to run locally or as a lightweight server; no cost. Great for prototypes. 
GitHub

FAISS (open-source) — library only (in-process index), very fast and free. Use when you want maximum control. 
GitHub
+1

Pinecone (managed) — has a free Starter plan good for prototyping (limits apply e.g., index size / read/write units). Use it if you want a managed vector DB without ops. 
Pinecone
+1

2) Embeddings (semantic vectors)

Local models (free) — sentence-transformers (all-MiniLM, etc.) run locally with zero API cost. Best for prototypes and privacy.

Hugging Face Inference — offers free tier/credits for serverless inference; good for occasional hosted model use. Watch quota limits. 
Hugging Face
+1

OpenAI — generally paid; limited trial credit or regional promotions may exist but don’t rely on free permanent plan. Check your account for any credits. 
OpenAI Developer Community
+1

3) LLM (planning / chat)

Hugging Face Hosted inference (free credits) or run a small local LLM (gpt4all/llama.cpp/quantized models) for planning prompts. HF free tier is convenient but limited; local models avoid costs. 
Hugging Face

4) Storage & DB

Postgres + PostGIS — fully open-source, free to run locally (or in any cloud free tier).

MinIO — S3-compatible object store you can run locally (free OSS).

TimescaleDB — free community version for time-series on a single node.

Redis — free to run locally for caching/session.

5) Satellite / raster

Sentinel (Copernicus) data is freely available (download and process locally). If you want processed commercial tiles (Planet/Maxar) those are paid — for a prototype use Sentinel free data or smaller-resolution public sources.

In [ ]:
Short “what to run now” checklist (copy/paste)

 docker-compose with Postgres+PostGIS, MinIO, Redis.

 Install Python deps: sentence-transformers, faiss-cpu (or chromadb), fastapi, boto3, psycopg2.

 Start with FAISS + sentence-transformers for embeddings & nearest-neighbor retrieval.

 If you want managed vectors: sign up Pinecone Starter (free) and test migration. 
Pinecone

 Sign up Hugging Face, redeem free inference credits to try hosted LLMs. 
Hugging Face